# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the [FAIR²](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. You will learn to load metadata, enumerate record sets and their fields, extract records, and perform exploratory analysis—all referencing entities by their Croissant `@id`s.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure mlcroissant and typical data libraries are installed
!pip install mlcroissant pandas matplotlib seaborn --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load metadata and show name/description
dataset = mlc.Dataset(croissant_url)
metadata_json = dataset.metadata.to_json()

print("Dataset Title:", dataset.metadata.name)
print("Description:", dataset.metadata.description)
print("Published on:", dataset.metadata.datePublished)
print("Cite as:", dataset.metadata.citeAs)
print("License:", dataset.metadata.license)

## 2. Data Overview
Review available record sets, their fields, and their Croissant `@id` values for navigation.

`mlcroissant` exposes the record sets defined in the Croissant schema. Each record set, field, and column is referenced by its `@id`, which we will use to explore and load data.

In [ ]:
# List all record sets by @id and show their fields by @id
record_sets = list(dataset.metadata.recordSets)
print(f"Found {len(record_sets)} record sets:")
for rs in record_sets:
    print(f"- Record Set @id: {rs['@id']}  |  Name: {rs['name']}")
    if rs.get('fields'):
        print("  Fields (by @id):")
        for fld in rs['fields']:
            fid = fld['@id']
            print(f"    - {fid}" + (f"  (name: {fld.get('name', '')})" if fld.get('name') else ""))
    print()

## 3. Data Extraction
Load data from a selected record set into a DataFrame for analysis. 
Use the record set and field `@id`s printed above. Here, we will load the main protein analysis table (the record set containing most protein results/data). 

*⚠️ If there are multiple record sets, you can tweak this code by selecting the `@id` corresponding to the table you're interested in.*


In [ ]:
# Select a record set @id to extract records (replace below if needed)
main_record_set_id = dataset.metadata.recordSets[0]['@id']
print(f"Using record set: {main_record_set_id}")

# Extract all records from this record set
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print(f"Loaded {len(df)} records. Columns (field @ids):")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering numeric fields (e.g., molecular weight, peptide counts), normalizing, and grouping by protein description or accession. All columns are referenced by their Croissant `@id` (see previous EDA cell output for options).


In [ ]:
# Choose a numeric field (e.g., 'cr:peptide_count' or actual ID found above)
# For demonstration, we will attempt to pick a numeric column from the extracted df columns by best-guess.
# If your schema gives different @id, adjust as necessary.
import numpy as np

candidate_numeric_cols = [col for col in df.columns if df[col].dtype in [np.float64, np.int64] or np.issubdtype(df[col].dtype, np.number)]
if not candidate_numeric_cols:
    # Try to coerce common likely-numeric columns
    for col in df.columns:
        if 'count' in col or 'MW' in col or 'abundance' in col or 'coverage' in col or col.startswith('cr:') or col.startswith('dv:'):
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            except Exception:
                pass
    candidate_numeric_cols = [col for col in df.columns if df[col].dtype in [np.float64, np.int64]]

if candidate_numeric_cols:
    numeric_field_id = candidate_numeric_cols[0]
else:
    # Fallback to first column
    numeric_field_id = df.columns[0]

print(f"Using numeric field @id for filtering and normalization: {numeric_field_id}")

# Set a threshold (example: field > 10)
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a likely field (such as accession or description, using @id)
possible_group_fields = [col for col in df.columns if 'accession' in col.lower() or 'description' in col.lower() or 'name' in col.lower()]
if possible_group_fields:
    group_field = possible_group_fields[0]
else:
    group_field = df.columns[0]  # default to something

if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
    print(f"Grouped data by {group_field}, showing mean {numeric_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and its normalized version, and a grouped summary if appropriate.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 5))
sns.histplot(filtered_df[numeric_field_id].dropna(), bins=30, color='skyblue', kde=True)
plt.title(f"Distribution of {numeric_field_id} (filtered > {threshold})")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

plt.figure(figsize=(10, 5))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), bins=30, color='salmon', kde=True)
plt.title(f"Normalized Distribution of {numeric_field_id} (filtered > {threshold})")
plt.xlabel(f"{numeric_field_id}_normalized")
plt.ylabel('Count')
plt.show()

# If group_field makes sense, show means as barplot
if 'grouped_df' in locals() and not grouped_df.empty:
    grouped_short = grouped_df.head(15)
    plt.figure(figsize=(10, 5))
    sns.barplot(y=group_field, x=numeric_field_id, data=grouped_short.reset_index())
    plt.title(f"Mean {numeric_field_id} by {group_field} (top 15)")
    plt.xlabel(f"Mean {numeric_field_id}")
    plt.ylabel(group_field)
    plt.show()

## 6. Conclusion
- We loaded the [FAIR²](https://doi.org/10.71728/senscience.vq4a-28xa) protein mass spectrometry dataset via its Croissant schema using `mlcroissant`.
- All dataset navigation referenced entities by their unique `@id`, enabling robust schema-based data handling.
- The notebook demonstrated how to enumerate record sets and fields, extract full records, perform filtering, normalization, and visual analytics using only the schema.

*You can adapt analysis and plotting by choosing different record sets, fields, or thresholds by their `@id` as required for your scientific workflow.*